In [1]:
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import sklearn
import pandas as pd
import os
import sys
import time
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

seed = 42
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)

cuda


In [2]:
import re
import unicodedata
from sklearn.model_selection import train_test_split

def unicode_to_ascii(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s)
                   if unicodedata.category(c) != 'Mn')
    
en_sentence = "May I borrow this book?"
sp_sentence = "¿Puedo tomar prestado este libro?"

print(unicode_to_ascii(en_sentence))
print(unicode_to_ascii(sp_sentence))

May I borrow this book?
¿Puedo tomar prestado este libro?


In [3]:
def preprocess_sentence(w):
    w = unicode_to_ascii(w.lower().strip())
    
    # creating a space between a word and the punctuation following it
    # eg: "he is a boy." => "he is a boy ." 
    # Reference:- https://stackoverflow.com/questions/3645931/python-padding-punctuation-with-white-spaces-keeping-punctuation
    w = re.sub(r"([?.!,¿])", r" \1 ", w)
    
    
    # replacing everything with space except (a-z, A-Z, ".", "?", "!", ",")
    w = re.sub(r"[^a-zA-Z?.!,¿¡]+", " ", w)
    w = re.sub(r'[" "]+', " ", w)
    
    w = w.rstrip().strip()
    
    # adding a start and an end token to the sentence
    # so that the model know when to start and stop predicting.
    w = '<start> ' + w + ' <end>'
    return w

print(preprocess_sentence(en_sentence))
print(preprocess_sentence(sp_sentence)) 
print(preprocess_sentence(sp_sentence).encode("utf-8"))


<start> may i borrow this book ? <end>
<start> ¿ puedo tomar prestado este libro ? <end>
b'<start> \xc2\xbf puedo tomar prestado este libro ? <end>'


In [4]:
a = [[1,2],[4,5],[7,8]]
sample, label = zip(*a)
print(sample)
print(label)

split_index = np.random.choice(a=["train", "test"], replace=True, p=[0.9, 0.1], size=100)
print(split_index)


(1, 4, 7)
(2, 5, 8)
['train' 'test' 'train' 'train' 'train' 'train' 'train' 'train' 'train'
 'train' 'train' 'test' 'train' 'train' 'train' 'train' 'train' 'train'
 'train' 'train' 'train' 'train' 'train' 'train' 'train' 'train' 'train'
 'train' 'train' 'train' 'train' 'train' 'train' 'test' 'test' 'train'
 'train' 'train' 'train' 'train' 'train' 'train' 'train' 'test' 'train'
 'train' 'train' 'train' 'train' 'train' 'test' 'train' 'test' 'train'
 'train' 'test' 'train' 'train' 'train' 'train' 'train' 'train' 'train'
 'train' 'train' 'train' 'train' 'train' 'train' 'test' 'train' 'train'
 'train' 'train' 'train' 'train' 'train' 'train' 'train' 'train' 'train'
 'train' 'train' 'train' 'train' 'train' 'train' 'train' 'train' 'train'
 'train' 'train' 'train' 'train' 'train' 'train' 'train' 'train' 'train'
 'train']


In [5]:
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

class LangPairDataset(Dataset):
    fpath = Path(r"./spa.txt") #数据文件路径
    cache_path = Path(r"./.cache/lang_pair.npy") #缓存文件路径
    split_index = np.random.choice(a=["train", "test"], replace=True, p=[0.9, 0.1], size=118964) #按照9:1划分训练集和测试集
    def __init__(self, mode="train", cache=False):
        if cache or not self.cache_path.exists():#如果没有缓存，或者缓存不存在，就处理一下数据
            self.cache_path.parent.mkdir(parents=True, exist_ok=True) #创建缓存文件夹，如果存在就忽略
            with open(self.fpath, "r", encoding="utf8") as file:
                lines = file.readlines()
                lang_pair = [[preprocess_sentence(w) for w in l.split('\t')]  for l in lines] #处理数据，变成list((trg, src))的形式
                trg, src = zip(*lang_pair) #分离出目标语言和源语言,如果要让英语作为src，把这里交换位置即可
                trg=np.array(trg) #转换为numpy数组
                src=np.array(src) #转换为numpy数组
                np.save(self.cache_path, {"trg": trg, "src": src})  #保存为npy文件,方便下次直接读取,不用再处理
        else:
            lang_pair = np.load(self.cache_path, allow_pickle=True).item() #读取npy文件，allow_pickle=True允许读取字典
            trg = lang_pair["trg"]
            src = lang_pair["src"]

        self.trg = trg[self.split_index == mode] #按照index拿到训练集的 标签语言 --英语
        self.src = src[self.split_index == mode] #按照index拿到训练集的源语言 --西班牙
        
    def __getitem__(self, index):
        return self.src[index], self.trg[index]
    
    def __len__(self):
        return len(self.src)

train_ds = LangPairDataset("train")
test_ds = LangPairDataset("test")

print(*train_ds[-1])

<start> si quieres sonar como un hablante nativo , debes estar dispuesto a practicar diciendo la misma frase una y otra vez de la misma manera en que un musico de banjo practica el mismo fraseo una y otra vez hasta que lo puedan tocar correctamente y en el tiempo esperado . <end> <start> if you want to sound like a native speaker , you must be willing to practice saying the same sentence over and over in the same way that banjo players practice the same phrase over and over until they can play it correctly and at the desired tempo . <end>


In [6]:
for pair in train_ds:
    print(pair[0])
    print(pair[1])
    break

print(len(train_ds[-1][0]))
print(len(train_ds[-1][1]))


<start> ve . <end>
<start> go . <end>
280
263


In [7]:
from collections import Counter

def get_word_index(ds,mode = "src",threshold = 2):
    word2index = {
        "[PAD]": 0,
        "[BOS]": 1,
        "[UNK]": 2,
        "[EOS]": 3,
    }
    index2word = {v: k for k, v in word2index.items()}
    index = len(index2word)
    threshold = 1
    word_list = " ".join([pair[0 if mode == "src" else 1] for pair in ds]).split()
    counter = Counter(word_list)
    print("word count:", len(counter))
    
    
    for token, count in counter.items():
        if count >= threshold:
            word2index[token] = index
            index2word[index] = token
            index += 1
    return word2index, index2word

src_word2index, src_index2word = get_word_index(train_ds, "src")
trg_word2index, trg_index2word = get_word_index(train_ds, "trg")

word count: 24004
word count: 12502


In [8]:
class Tokenizer:
    def __init__(self, word2index, index2word, max_length=500,pad_index=0,bos_index=1,eos_index=3,unk_index=2):
        self.word2index = word2index
        self.index2word = index2word
        self.max_length = max_length
        self.pad_index = pad_index
        self.bos_index = bos_index
        self.eos_index = eos_index
        self.unk_index = unk_index
    
    def encode(self, text_list, padding_first=False, add_bos=True, add_eos=True, return_mask=False):
        max_len = min(self.max_length, add_eos + add_bos + max(len(text) for text in text_list))
        indices_list = []
        for text in text_list:
            indices = [self.word2index.get(word, self.unk_index) for word in text[:max_len - add_bos - add_eos]]
            if add_bos:
                indices = [self.bos_index] + indices
            if add_eos:
                indices = indices + [self.eos_index]
            if padding_first:
                indices = [self.pad_index] * (max_len - len(indices)) + indices
            else:
                indices = indices + [self.pad_index] * (max_len - len(indices))
            indices_list.append(indices)
        input_ids = torch.tensor(indices_list)
        masks = (input_ids == self.pad_index).to(dtype=torch.int64)
        return input_ids if not return_mask else (input_ids, masks)
        
    def decode(self, indices_list, remove_pad=True, remove_bos=True, remove_eos=True, split=False):
        text_list = []
        for indices in indices_list:
            text = []
            for index in indices:
                word = self.index2word.get(index, "[UNK]")
                if remove_bos and word == "[BOS]":
                    continue
                if remove_eos and word == "[EOS]":
                    break
                if remove_pad and word == "[PAD]":
                    break
                text.append(word)
            text_list.append(" ".join(text) if not split else text)
        return text_list

src_tokenizer = Tokenizer(word2index=src_word2index, index2word=src_index2word)
trg_tokenizer = Tokenizer(word2index=trg_word2index, index2word=trg_index2word) 

raw_text = ["hello world".split(), "tokenize text datas with batch".split(), "this is a test".split()]
indices,mask = trg_tokenizer.encode(raw_text, padding_first=False, add_bos=True, add_eos=True,return_mask=True)

for raw,idx,msk in zip(raw_text,indices,mask):
    print(raw,idx,msk)


decoded_text = trg_tokenizer.decode(indices)
for decode in decoded_text:
    print(decode)

['hello', 'world'] tensor([   1,   18, 3220,    3,    0,    0,    0]) tensor([0, 0, 0, 0, 1, 1, 1])
['tokenize', 'text', 'datas', 'with', 'batch'] tensor([   1,    2, 3880,    2,  554,    2,    3]) tensor([0, 0, 0, 0, 0, 0, 0])
['this', 'is', 'a', 'test'] tensor([   1,  119,  237,  105, 2898,    3,    0]) tensor([0, 0, 0, 0, 0, 0, 1])
[UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK]
[UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK]
[UNK] [UNK] [UNK] [UNK] [UNK] [UNK] [UNK]


In [9]:
def collate_fn(batch):
    src_words = [pair[0].split() for pair in batch]
    trg_words = [pair[1].split() for pair in batch]
    
    encoder_inputs, encoder_inputs_mask = src_tokenizer.encode(
        src_words,
        padding_first = True,
        add_bos=True,
        add_eos=True,
        return_mask=True,
    )
    
    decoder_inputs = trg_tokenizer.encode(
        trg_words,
        padding_first = False,
        add_bos=True,
        add_eos=False,
        return_mask=False,
    )
    
    decoder_labels, decoder_labels_mask = trg_tokenizer.encode(
        trg_words,
        padding_first = False,
        add_bos=False,
        add_eos=True,
        return_mask=True,
    )
    return {
        "encoder_inputs": encoder_inputs.to(device),
        "encoder_inputs_mask": encoder_inputs_mask.to(device),
        "decoder_inputs": decoder_inputs.to(device),
        "decoder_labels": decoder_labels.to(device),
        "decoder_labels_mask": decoder_labels_mask.to(device),
    }



sample_dl = DataLoader(train_ds, batch_size=2, shuffle=True, collate_fn=collate_fn)

for batch in sample_dl:
    for src, trg in batch.items():
        print(src)
        print(trg)
    break

encoder_inputs
tensor([[   0,    1,    4,   58,   70, 1106,  322,   53,    6,    7,    3],
        [   1,    4,   97, 5757,   53, 2714,  511, 3869,    6,    7,    3]],
       device='cuda:0')
encoder_inputs_mask
tensor([[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], device='cuda:0')
decoder_inputs
tensor([[   1,    4,   19,   34,  518,   32, 1090, 1579,    6,    7,    0],
        [   1,    4,   49, 2978,  691, 5402, 2240,  636,   31,    6,    7]],
       device='cuda:0')
decoder_labels
tensor([[   4,   19,   34,  518,   32, 1090, 1579,    6,    7,    3,    0],
        [   4,   49, 2978,  691, 5402, 2240,  636,   31,    6,    7,    3]],
       device='cuda:0')
decoder_labels_mask
tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], device='cuda:0')


In [10]:
#Encoder
class  Encoder(nn.Module):
    def __init__(self,vocab_size , emb_dim=256, hid_dim=1024, n_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.gru = nn.GRU(emb_dim, hid_dim, num_layers=n_layers, batch_first=True)
        
    def forward(self, src):
        # src: [batch_size, src_len]
        embedded = self.embedding(src) # embedded: [batch_size, src_len, emb_dim]
        outputs, hidden = self.gru(embedded) # outputs: [batch_size, src_len, hid_dim], hidden: [n_layers, batch_size, hid_dim]
        return outputs, hidden

In [11]:
#实例
encoder = Encoder(vocab_size=100, emb_dim=256, hid_dim=1024, n_layers=4)
encoder_inputs = torch.randint(0, 100, (2, 50))
encoder_outputs, encoder_hidden = encoder(encoder_inputs)
print(encoder_outputs.shape)
print(encoder_hidden.shape)
print(encoder_outputs[:,-1,:])
print(encoder_hidden[-1,:,:])


torch.Size([2, 50, 1024])
torch.Size([4, 2, 1024])
tensor([[-0.0636,  0.0092,  0.0103,  ..., -0.0153, -0.0004, -0.0341],
        [-0.0175,  0.0283,  0.0105,  ...,  0.0142,  0.0282, -0.0413]],
       grad_fn=<SliceBackward0>)
tensor([[-0.0636,  0.0092,  0.0103,  ..., -0.0153, -0.0004, -0.0341],
        [-0.0175,  0.0283,  0.0105,  ...,  0.0142,  0.0282, -0.0413]],
       grad_fn=<SliceBackward0>)


In [12]:
query1 = torch.randn(2, 1024)
query1.unsqueeze(-2).shape

torch.Size([2, 1, 1024])

In [13]:
#Bahdanau Attention
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size=1024):
        super().__init__()
        self.Wk = nn.Linear(hidden_size, hidden_size)
        self.Wq = nn.Linear(hidden_size, hidden_size)
        self.V = nn.Linear(hidden_size, 1)
        
    def forward(self, query, keys, values, attn_mask=None):
        # query: (batch_size, hidden_size)
        # keys: (batch_size, seq_len, hidden_size)
        # values: (batch_size, seq_len, hidden_size)
        # attn_weights: (batch_size, seq_len, 1)
        scores = self.V(F.tanh(self.Wk(keys) + self.Wq(query.unsqueeze(-2))))
        if attn_mask is not None:
            attn_mask = (attn_mask.unsqueeze(-1)) * -1e16
            scores += attn_mask
        attn_weights = F.softmax(scores, dim=-2)
        # context: (batch_size, hidden_size)
        context = torch.mul(attn_weights, values).sum(dim=-2)
        return context, attn_weights

a = torch.randn(2, 3)
b = torch.randn(2, 3)
c = torch.mul(a, b)
print(c.shape)

torch.Size([2, 3])


In [14]:
attention = BahdanauAttention(hidden_size=1024)
query = torch.randn(2, 1024)
keys = torch.randn(2, 50, 1024)
values = torch.randn(2, 50, 1024)
attn_mask = torch.randint(0, 2, (2, 50))
context, attn_weights = attention(query, keys, values, attn_mask)
print(f'context.shape = {context.shape}')
print(f'attn_weights.shape = {attn_weights.shape}')

context.shape = torch.Size([2, 1024])
attn_weights.shape = torch.Size([2, 50, 1])


In [15]:
#Decoder
class Decoder(nn.Module):
    def __init__(self,vocab_size , emb_dim=256, hid_dim=1024, n_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.gru = nn.GRU(emb_dim + hid_dim, hid_dim, num_layers=n_layers, batch_first=True)
        self.fc = nn.Linear(hid_dim, vocab_size)
        self.dropout = nn.Dropout(0.6)
        self.attention = BahdanauAttention(hid_dim)
    
    def forward(self, decoder_input, hidden, encoder_outputs, attn_mask=None):
        assert len(decoder_input.shape) == 2 and decoder_input.shape[-1] == 1, f"decoder_input.shape = {decoder_input.shape} is not valid"
        assert len(hidden.shape) == 2, f"hidden.shape = {hidden.shape} is not valid"
        assert len(encoder_outputs.shape) == 3, f"encoder_outputs.shape = {encoder_outputs.shape} is not valid"
        
        context_vector, attention_score = self.attention(query=hidden, keys=encoder_outputs, values=encoder_outputs, attn_mask=attn_mask)
        embeds = self.embedding(decoder_input)
        embeds = torch.cat((context_vector.unsqueeze(-2),embeds), dim=-1)
        hidden = hidden.unsqueeze(0)
        seq_output, hidden = self.gru(embeds, hidden)
        logits = self.fc(self.dropout(seq_output))
        return logits, hidden, attention_score

In [16]:
vocab_size = 100
batch_size = 2
seq_len = 10
embedding_dim = 256
hidden_dim = 1024

decoder = Decoder(
    vocab_size=vocab_size,
    emb_dim=embedding_dim,
    hid_dim=hidden_dim
)

decoder_input = torch.randint(0, vocab_size, (batch_size, 1))
hidden = torch.randn(batch_size, hidden_dim)
encoder_outputs = torch.randn(batch_size, seq_len, hidden_dim)
attn_mask = None

logits, hidden_out, attention_score = decoder(decoder_input, hidden, encoder_outputs, attn_mask)
print(logits.shape)
print(hidden_out.shape)
print(attention_score.shape)

torch.Size([2, 1, 100])
torch.Size([1, 2, 1024])
torch.Size([2, 10, 1])


In [17]:
class Sequence2Sequence(nn.Module):
    def __init__(
        self,
        src_vocab_size, #输入词典大小
        trg_vocab_size, #输出词典大小
        encoder_embedding_dim=256,
        encoder_hidden_dim=1024, #encoder_hidden_dim和decoder_hidden_dim必须相同，是因为BahdanauAttention设计的
        encoder_num_layers=1,
        decoder_embedding_dim=256,
        decoder_hidden_dim=1024,
        decoder_num_layers=1,
        bos_idx=1,
        eos_idx=3,
        max_length=512,
        ):
        super().__init__()
        self.bos_idx = bos_idx
        self.eos_idx = eos_idx
        self.max_length = max_length
        self.encoder = Encoder(
            vocab_size=src_vocab_size,
            emb_dim=encoder_embedding_dim,
            hid_dim=encoder_hidden_dim,
            n_layers=encoder_num_layers,
            )
        self.decoder = Decoder(
            vocab_size=trg_vocab_size,
            emb_dim=decoder_embedding_dim,
            hid_dim=decoder_hidden_dim,
            n_layers=decoder_num_layers,
            )
    
    def forward(self, *, encoder_inputs, decoder_inputs, attn_mask=None):
        encoder_outputs, hidden = self.encoder(encoder_inputs)
        bs, seq_len = decoder_inputs.shape
        logits_list = []
        scores_list = []
        for i in range(seq_len):
            logits, hidden, scores = self.decoder(
                decoder_inputs[:, i:i+1],
                hidden[-1],
                encoder_outputs,
                attn_mask=attn_mask
                )
            logits_list.append(logits)
            scores_list.append(scores)
        return torch.cat(logits_list, dim=-2), torch.cat(scores_list, dim=-1)
    
    @torch.no_grad()
    def infer(self, encoder_input, attn_mask=None):
        encoder_outputs, hidden = self.encoder(encoder_input)

        decoder_input = torch.Tensor([self.bos_idx]).reshape(1, 1).to(dtype=torch.int64)
        decoder_pred = None
        pred_list = []
        scores_list = []
        for i in range(self.max_length):
            logits, hidden, score = self.decoder(
                decoder_input,
                hidden[-1],
                encoder_outputs,
                attn_mask=attn_mask
                )
            decoder_pred = logits.argmax(dim=-1)
            decoder_input = decoder_pred
            pred_list.append(decoder_pred.reshape(-1).item())
            scores_list.append(score)
            
            if decoder_pred == self.eos_idx:
                break
        
        return pred_list, torch.cat(scores_list, dim=-1)

In [18]:
model = Sequence2Sequence(src_vocab_size=len(src_word2index), trg_vocab_size=len(trg_word2index))

encoder_inputs = torch.randint(0, 100, (2, 50))
decoder_inputs = torch.randint(0, 100, (2, 60))
attn_mask = torch.randint(0, 2, (2, 50))
logits, scores = model(encoder_inputs=encoder_inputs, decoder_inputs=decoder_inputs, attn_mask=attn_mask)
print(logits.shape, scores.shape)

torch.Size([2, 60, 12506]) torch.Size([2, 50, 60])


In [19]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'The model has {count_parameters(model):,} trainable parameters')

model = Sequence2Sequence(src_vocab_size=len(src_word2index), trg_vocab_size=len(trg_word2index))
print(count_parameters(model))

The model has 35,288,795 trainable parameters
35288795


In [20]:
#cross_entropy_with_padding
def cross_entropy_with_padding(logits, labels, padding_mask=None):
    """
    计算交叉熵损失，忽略填充索引
    :param logits: 模型输出，形状为 (batch_size, seq_len, num_classes)
    :param labels: 真实标签，形状为 (batch_size, seq_len)
    :param padding_mask: 填充掩码，形状为 (batch_size, seq_len)，默认值为 None
    :return: 损失值
    """
    batch_size, seq_len, num_classes = logits.shape
    loss = F.cross_entropy(logits.reshape(batch_size * seq_len, num_classes),
                           labels.reshape(-1),
                           reduction='none')
    if padding_mask is not None:
        padding_mask = 1 - padding_mask.reshape(-1)
        loss = torch.mul(loss, padding_mask).sum() / padding_mask.sum()
    else:
        loss = loss.mean()
    return loss


In [21]:
#callback函数
from torch.utils.tensorboard import SummaryWriter

class TensorBoardCallback:
    """
    TensorBoard Callback
    """
    def __init__(self, log_dir,flush_secs=10):
        self.writer = SummaryWriter(log_dir=log_dir, flush_secs=flush_secs)
    def draw_model(self, model, input_shape):
        self.writer.add_graph(model, input_to_model=torch.randn(input_shape))

    def add_loss_scalars(self, step, loss, val_loss):
        self.writer.add_scalars(
            main_tag="training/loss",
            tag_scalar_dict={"loss": loss, "val_loss": val_loss},
            global_step=step,
            )

    def add_acc_scalars(self, step, acc, val_acc):
        self.writer.add_scalars(
            main_tag="training/accuracy",
            tag_scalar_dict={"accuracy": acc, "val_accuracy": val_acc},
            global_step=step,
        )

    def add_lr_scalars(self, step, learning_rate):
        self.writer.add_scalars(
            main_tag="training/learning_rate",
            tag_scalar_dict={"learning_rate": learning_rate},
            global_step=step,

        )

    def __call__(self, step, **kwargs):
        # add loss
        loss = kwargs.pop("loss", None)
        val_loss = kwargs.pop("val_loss", None)
        if loss is not None and val_loss is not None:
            self.add_loss_scalars(step, loss, val_loss)
        # add acc
        acc = kwargs.pop("acc", None)
        val_acc = kwargs.pop("val_acc", None)
        if acc is not None and val_acc is not None:
            self.add_acc_scalars(step, acc, val_acc)
        # add lr
        learning_rate = kwargs.pop("lr", None)
        if learning_rate is not None:
            self.add_lr_scalars(step, learning_rate)


In [22]:
class SaveCallback:
    """
    Save Callback
    """
    def __init__(self, save_dir, save_step = 5000, save_best_only = True):
        self.save_dir = save_dir
        self.save_step = save_step
        self.save_best_only = save_best_only
        self.best_metric = - np.inf
        
        if not os.path.exists(self.save_dir):
            os.mkdir(self.save_dir)
    
    def __call__(self, step, state_dict, metric = None):
        if step % self.save_step > 0:
            return
        
        if self.save_best_only:
            assert metric is not None
            if metric >= self.best_metric:
                # save checkpoints
                torch.save(state_dict, os.path.join(self.save_dir, "best.ckpt"))
                # update best metrics
                self.best_metric = metric
        else:
            torch.save(state_dict, os.path.join(self.save_dir, f"{step}.ckpt"))

In [23]:
class EarlyStoppingCallback:
    def __init__(self, patience=10, min_delta=0.01):
        """
        Early stopping callback.
        Stop training if validation loss doesn't improve after patience epochs.

        Note:
            This callback is used to stop training when a monitored quantity has stopped improving.
        Args:
            patience (int, optional): Number of epochs to wait before stopping. Defaults to 10.
            min_delta (float, optional): Minimum change in the monitored quantity to qualify as an improvement. Defaults to 0.
        """
        self.patience = patience
        self.min_delta = min_delta
        self.best_metric = -np.inf
        self.counter = 0
    
    def __call__(self, metric):
        if metric >= self.best_metric + self.min_delta:
            self.best_metric = metric
            self.counter = 0
        else:
            self.counter += 1
    
    @property
    def early_stop(self):
        return self.counter >= self.patience

In [24]:
#train and valuate
def evaluate(model, dataloader, loss_fct):
    loss_list = []
    for batch in dataloader:
        encoder_inputs = batch["encoder_inputs"]
        encoder_inputs_mask = batch["encoder_inputs_mask"]
        decoder_inputs = batch["decoder_inputs"]
        decoder_labels = batch["decoder_labels"]
        decoder_labels_mask = batch["decoder_labels_mask"]
        
        logits, _ = model(
            encoder_inputs=encoder_inputs,
            decoder_inputs=decoder_inputs,
            attn_mask=encoder_inputs_mask
            )
        loss = loss_fct(logits, decoder_labels, padding_mask=decoder_labels_mask)
        loss_list.append(loss.cpu().item())
        
    return np.mean(loss_list)

In [25]:
def training(
    model,
    train_loader,
    val_loader,
    epoch,
    loss_fct,
    optimizer,
    tensorboard_callback=None,
    save_ckpt_callback=None,
    early_stop_callback=None,
    eval_step=500,
    ):
    record_dict = {
        "train": [],
        "val": []
    }
    
    global_step = 1
    model.train()
    with tqdm(total=len(train_loader) * epoch) as pbar:
        for epoch_id in range(epoch):
            for batch in train_loader:
                encoder_inputs = batch["encoder_inputs"]
                encoder_inputs_mask = batch["encoder_inputs_mask"]
                decoder_inputs = batch["decoder_inputs"]
                decoder_labels = batch["decoder_labels"]
                decoder_labels_mask = batch["decoder_labels_mask"]

                optimizer.zero_grad()
                
                logits, _ = model(
                    encoder_inputs=encoder_inputs,
                    decoder_inputs=decoder_inputs,
                    attn_mask=encoder_inputs_mask
                    )
                loss = loss_fct(logits, decoder_labels, padding_mask=decoder_labels_mask)
                
                loss.backward()
                
                optimizer.step()
                
                loss = loss.cpu().item()
                record_dict["train"].append({
                        "loss": loss, "step": global_step
                    })
                
                if global_step % eval_step == 0:
                    model.eval()
                    val_loss = evaluate(model, val_loader, loss_fct)
                    record_dict["val"].append({
                        "loss": val_loss, "step": global_step
                    })
                    model.train()
                    
                    if tensorboard_callback is not None:
                        tensorboard_callback(
                                global_step,
                                loss=loss, val_loss=val_loss,
                                lr=optimizer.param_groups[0]["lr"],
                                )
                    
                    if save_ckpt_callback is not None:
                        save_ckpt_callback(
                            step=global_step,
                            state_dict=model.state_dict(),
                            metric=-val_loss,
                            )
                        
                    if early_stop_callback is not None:
                        early_stop_callback(-val_loss)
                        if early_stop_callback.early_stop:
                            print(f"Early stop at epoch {epoch_id} / global_step {global_step}")
                            return record_dict
                    print(f"step: {global_step}, val_loss: {val_loss:.4f}")
                global_step += 1
                pbar.update(1)
            pbar.set_postfix({"epoch": epoch_id, "loss": loss, "val_loss": val_loss})

    return record_dict

epoch = 20
batch_size = 64

model = Sequence2Sequence(src_vocab_size=len(src_word2index), trg_vocab_size=len(trg_word2index))
train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
test_dl = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

loss_fct = cross_entropy_with_padding
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

if not os.path.exists("runs"):
    os.mkdir("runs")
exp_name = "translate-seq2seq"

if not os.path.exists("checkpoints"):
    os.makedirs("checkpoints")
save_ckpt_callback = SaveCallback(
    f"checkpoints/{exp_name}", save_step=200, save_best_only=True)

early_stop_callback = EarlyStoppingCallback(patience=5)

model = model.to(device)

In [ ]:
record = training(
    model,
    train_dl,
    test_dl,
    epoch,
    loss_fct,
    optimizer,
    tensorboard_callback=None,
    save_ckpt_callback=save_ckpt_callback,
    early_stop_callback=early_stop_callback,
    eval_step=200
    )

  0%|          | 0/33520 [00:00<?, ?it/s]

step: 200, val_loss: 2.5973
step: 400, val_loss: 2.1019
step: 600, val_loss: 1.8128
step: 800, val_loss: 1.6423
step: 1000, val_loss: 1.5195
step: 1200, val_loss: 1.4210
step: 1400, val_loss: 1.3596
step: 1600, val_loss: 1.3142
step: 1800, val_loss: 1.2561
step: 2000, val_loss: 1.2351
step: 2200, val_loss: 1.2012


In [ ]:
plt.plot([i["step"] for i in record["train"]], [i["loss"] for i in record["train"]], label="train")
plt.plot([i["step"] for i in record["val"]], [i["loss"] for i in record["val"]], label="val")
plt.grid()
plt.show()

In [ ]:
!cp checkpoints/translate-seq2seq/best.ckpt .

In [ ]:
model = Sequence2Sequence(len(src_word2index), len(trg_word2index)) #这里初始化的模型结构，必须和训练时初始化的保持一致
model.load_state_dict(torch.load(f"best.ckpt", weights_only=True,map_location="cpu"))

class Translator:
    def __init__(self, model, src_tokenizer, trg_tokenizer):
        self.model = model
        self.model.eval() # 切换到验证模式
        self.src_tokenizer = src_tokenizer
        self.trg_tokenizer = trg_tokenizer

    def draw_attention_map(self, scores, src_words_list, trg_words_list):
        """绘制注意力热力图

        Args:
            - scores (numpy.ndarray): shape = [source sequence length, target sequence length]
        """
        plt.matshow(scores.T, cmap='viridis') # 注意力矩阵,显示注意力分数值
        # 获取当前的轴
        ax = plt.gca()

        # 设置热图中每个单元格的分数的文本
        for i in range(scores.shape[0]): #输入
            for j in range(scores.shape[1]): #输出
                ax.text(j, i, f'{scores[i, j]:.2f}',  # 格式化数字显示
                               ha='center', va='center', color='k')

        plt.xticks(range(scores.shape[0]), src_words_list)
        plt.yticks(range(scores.shape[1]), trg_words_list)
        plt.show()

    def __call__(self, sentence):
        sentence = preprocess_sentence(sentence) # 预处理句子，标点符号处理等
        encoder_input, attn_mask = self.src_tokenizer.encode(
            [sentence.split()],
            padding_first=True,
            add_bos=True,
            add_eos=True,
            return_mask=True,
            ) # 对输入进行编码，并返回encode_piadding_mask
        encoder_input = torch.Tensor(encoder_input).to(dtype=torch.int64) # 转换成tensor

        preds, scores = model.infer(encoder_input=encoder_input, attn_mask=attn_mask) #预测

        trg_sentence = self.trg_tokenizer.decode([preds], split=True, remove_eos=False)[0] #通过tokenizer转换成文字

        src_decoded = self.src_tokenizer.decode(
            encoder_input.tolist(),
            split=True,
            remove_bos=False,
            remove_eos=False
            )[0] #对输入编码id进行解码，转换成文字,为了画图

        self.draw_attention_map(
            scores.squeeze(0).numpy(),
            src_decoded, # 注意力图的源句子
            trg_sentence # 注意力图的目标句子
            )
        return " ".join(trg_sentence[:-1])

In [ ]:
translator = Translator(model.cpu(), src_tokenizer, trg_tokenizer)
translator('hace mucho frio aqui .')

In [ ]:
translator(u'¿ A dónde iremos después ?')